# Alarm systems  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 2.4.b: Alarm event classifier

**Skill:** Detect Attacks · **Maps to:** 2.4.B · **Language:** Python 3 (pytest)

## Problem
You are building the triage logic for an alarm control panel at Crestfield Analytics. Given a
timestamp-ordered stream of sensor events (one per zone trip), classify each event using the
arm-state, exit-delay, and cross-zone verification rules taught in the lesson:

- **SUPPRESSED**, the system was `"DISARMED"` when the event occurred (does not apply to panic
  buttons, see below).
- **FALSE_ALARM**, the event's sensor is `"contact"` or `"glass_break"`, the system was armed,
  and the event happened less than `exit_delay_sec` seconds after the zone was armed (still inside
  the exit-delay grace period).
- **LOCAL_ONLY**, a real trip (armed, past the exit delay) that has no corroborating trip in a
  *different* zone within `verify_window_sec` seconds. Only the local siren sounds.
- **DISPATCH**, either (a) the sensor is `"panic"` (a panic/duress button always dispatches,
  regardless of arm state or timing), or (b) a real trip that has a corroborating real trip in a
  different zone within `verify_window_sec` seconds (cross-zone verification).

## Function contract
Implement `solve(events, exit_delay_sec=45, verify_window_sec=120)` in the code editor (see
the Reference section below).

## Input
- `events`: `list[dict]`, each with:
  - `"ts"`: `int`, Unix timestamp (seconds) the event occurred
  - `"zone"`: `str`, zone/door identifier, e.g. `"front-door"`
  - `"sensor"`: `str`, one of `"contact"`, `"glass_break"`, `"panic"`
  - `"arm_state"`: `str`, one of `"DISARMED"`, `"ARMED_AWAY"`, `"ARMED_STAY"`
  - `"since_armed_sec"`: `int`, seconds elapsed between arming and this event (ignored for
    `"panic"` and when `arm_state == "DISARMED"`)
- `exit_delay_sec`: `int`, default `45`, grace period after arming.
- `verify_window_sec`: `int`, default `120`, max gap (in `ts`) between two different-zone real
  trips for cross-zone verification to apply.

## Output
A `list[str]` of classifications, **one per input event, in the same order as the input**. Each
entry is one of `"SUPPRESSED"`, `"FALSE_ALARM"`, `"LOCAL_ONLY"`, `"DISPATCH"`.

## Examples
```python
events = [
    {"ts": 1000, "zone": "front-door", "sensor": "contact",
     "arm_state": "ARMED_AWAY", "since_armed_sec": 10},
]
solve(events)  ->  ["FALSE_ALARM"]

events2 = [
    {"ts": 1000, "zone": "z1", "sensor": "panic",
     "arm_state": "DISARMED", "since_armed_sec": 0},
]
solve(events2)  ->  ["DISPATCH"]

events3 = [
    {"ts": 1000, "zone": "z1", "sensor": "contact",
     "arm_state": "ARMED_AWAY", "since_armed_sec": 300},
    {"ts": 1050, "zone": "z2", "sensor": "glass_break",
     "arm_state": "ARMED_AWAY", "since_armed_sec": 305},
]
solve(events3)  ->  ["DISPATCH", "DISPATCH"]
```

## Constraints
- Deterministic, offline, standard library only.
- Events may be given in any timestamp order in the list; the output order must match the
  **input** order (do not re-sort the output).
- Cross-zone verification: two real trips (armed, past exit delay) in **different** zones with
  `abs(ts_a - ts_b) <= verify_window_sec` corroborate each other. A real trip may be corroborated
  by more than one other zone; any single qualifying pair is enough to promote it to `DISPATCH`.
  Same-zone repeats never corroborate each other.
- `since_armed_sec >= exit_delay_sec` counts as past the exit delay (boundary is inclusive on the
  "past delay" side, i.e. exactly `exit_delay_sec` is a real trip, not a false alarm).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(events: list[dict], exit_delay_sec: int = 45, verify_window_sec: int = 120) -> list[str]
```

### Input (`events`)
`list[dict]` where each dict has:
- `"ts"`: `int`, Unix timestamp in seconds
- `"zone"`: `str`, zone/door identifier
- `"sensor"`: `str`, one of `"contact"`, `"glass_break"`, `"panic"`
- `"arm_state"`: `str`, one of `"DISARMED"`, `"ARMED_AWAY"`, `"ARMED_STAY"`
- `"since_armed_sec"`: `int`, seconds since the zone was armed when this event occurred (only
  meaningful when `arm_state != "DISARMED"` and `sensor != "panic"`)

### Input (other parameters)
- `exit_delay_sec`: `int`, default `45`, an armed contact/glass_break trip with
  `since_armed_sec < exit_delay_sec` is a false alarm.
- `verify_window_sec`: `int`, default `120`, max `ts` gap between two different-zone real trips
  for cross-zone verification.

### Output (`result`)
A `list[str]` the same length as `events`, in the **same order as the input**, where each element
is one of `"SUPPRESSED"`, `"FALSE_ALARM"`, `"LOCAL_ONLY"`, `"DISPATCH"`.

### Classification rules
1. **Panic override:** if `sensor == "panic"`, classification is always `"DISPATCH"`, regardless
   of `arm_state` or `since_armed_sec`.
2. **Suppressed:** else if `arm_state == "DISARMED"`, classification is `"SUPPRESSED"`.
3. **False alarm:** else if `since_armed_sec < exit_delay_sec`, classification is `"FALSE_ALARM"`.
4. **Real trip, verification:** otherwise the event is a "real trip" candidate. A real trip is
   promoted to `"DISPATCH"` if there exists at least one other real-trip event (per rules 1-4,
   excluding panic events) in a **different** `zone` with `abs(ts_i - ts_j) <= verify_window_sec`.
   Otherwise it is `"LOCAL_ONLY"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output order matches input order exactly (do not sort by timestamp for output).
- Same-zone events never corroborate each other for cross-zone verification.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 2.4.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List, Dict


def solve(events: List[Dict], exit_delay_sec: int = 45, verify_window_sec: int = 120) -> List[str]:
    """Classify each alarm event as SUPPRESSED, FALSE_ALARM, LOCAL_ONLY, or DISPATCH.

    Rules (see the Reference section below):
      1. sensor == "panic"          -> DISPATCH (always, overrides everything else)
      2. arm_state == "DISARMED"    -> SUPPRESSED
      3. since_armed_sec < exit_delay_sec -> FALSE_ALARM
      4. otherwise a "real trip": DISPATCH if another real trip in a DIFFERENT zone
         occurs within verify_window_sec seconds, else LOCAL_ONLY.
    """
    n = len(events)
    base = [None] * n  # per-index: "PANIC", "SUPPRESSED", "FALSE_ALARM", or "REAL"

    for i, e in enumerate(events):
        if e["sensor"] == "panic":
            base[i] = "PANIC"
        elif e["arm_state"] == "DISARMED":
            base[i] = "SUPPRESSED"
        elif e["since_armed_sec"] < exit_delay_sec:
            base[i] = "FALSE_ALARM"
        else:
            base[i] = "REAL"

    real_indices = [i for i in range(n) if base[i] == "REAL"]

    result = [""] * n
    for i, e in enumerate(events):
        if base[i] == "PANIC":
            result[i] = "DISPATCH"
        elif base[i] == "SUPPRESSED":
            result[i] = "SUPPRESSED"
        elif base[i] == "FALSE_ALARM":
            result[i] = "FALSE_ALARM"
        else:
            corroborated = False
            for j in real_indices:
                if j == i:
                    continue
                if events[j]["zone"] == e["zone"]:
                    continue
                if abs(events[j]["ts"] - e["ts"]) <= verify_window_sec:
                    corroborated = True
                    break
            result[i] = "DISPATCH" if corroborated else "LOCAL_ONLY"

    return result


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

BASE = 1_700_000_000


def _e(ts, zone, sensor, arm_state, since_armed_sec):
    return {
        "ts": ts,
        "zone": zone,
        "sensor": sensor,
        "arm_state": arm_state,
        "since_armed_sec": since_armed_sec,
    }


# ---- visible cases -------------------------------------------------------

def test_false_alarm_inside_exit_delay():
    """Contact trip 10s after arming, well inside default 45s exit delay."""
    events = [_e(BASE, "front-door", "contact", "ARMED_AWAY", 10)]
    assert impl.solve(events) == ["FALSE_ALARM"]


def test_suppressed_when_disarmed():
    """Contact trip while disarmed is SUPPRESSED."""
    events = [_e(BASE, "back-window", "contact", "DISARMED", 0)]
    assert impl.solve(events) == ["SUPPRESSED"]


def test_panic_dispatch_even_disarmed():
    """Panic button always DISPATCH, even when arm_state is DISARMED."""
    events = [_e(BASE, "reception", "panic", "DISARMED", 0)]
    assert impl.solve(events) == ["DISPATCH"]


def test_real_trip_no_corroboration_local_only():
    """A real trip alone in the batch, past exit delay, with no other zone -> LOCAL_ONLY."""
    events = [_e(BASE, "garage", "glass_break", "ARMED_AWAY", 900)]
    assert impl.solve(events) == ["LOCAL_ONLY"]


def test_cross_zone_verification_dispatch():
    """Two real trips in different zones within verify_window_sec -> both DISPATCH."""
    events = [
        _e(BASE, "z1", "contact", "ARMED_AWAY", 300),
        _e(BASE + 50, "z2", "glass_break", "ARMED_AWAY", 305),
    ]
    assert impl.solve(events) == ["DISPATCH", "DISPATCH"]


def test_same_zone_repeats_do_not_corroborate():
    """Two real trips in the SAME zone within the window do not verify each other."""
    events = [
        _e(BASE, "z1", "contact", "ARMED_AWAY", 300),
        _e(BASE + 50, "z1", "contact", "ARMED_AWAY", 305),
    ]
    assert impl.solve(events) == ["LOCAL_ONLY", "LOCAL_ONLY"]


def test_empty_events():
    """Empty stream returns empty list."""
    assert impl.solve([]) == []


def test_mixed_batch_order_preserved():
    """Output order matches input order, not chronological or classification order."""
    events = [
        _e(BASE + 1000, "z9", "glass_break", "ARMED_AWAY", 900),   # LOCAL_ONLY (alone)
        _e(BASE, "front-door", "contact", "ARMED_AWAY", 5),        # FALSE_ALARM
        _e(BASE + 2000, "vault", "panic", "ARMED_STAY", 0),        # DISPATCH
    ]
    assert impl.solve(events) == ["LOCAL_ONLY", "FALSE_ALARM", "DISPATCH"]


# ---- hidden / edge cases -------------------------------------------------

def test_boundary_exit_delay_below_is_false_alarm():
    """44s (one below default 45s exit_delay_sec) is still FALSE_ALARM."""
    events = [_e(BASE, "lobby", "contact", "ARMED_AWAY", 44)]
    assert impl.solve(events) == ["FALSE_ALARM"]


def test_boundary_exit_delay_at_threshold_is_real():
    """Exactly 45s (== exit_delay_sec) counts as past the exit delay -> real trip -> LOCAL_ONLY alone."""
    events = [_e(BASE, "server-room", "contact", "ARMED_AWAY", 45)]
    assert impl.solve(events) == ["LOCAL_ONLY"]


def test_boundary_verify_window_exactly_at_limit():
    """Gap of exactly verify_window_sec (120s default) between different zones still verifies."""
    events = [
        _e(BASE, "a1", "contact", "ARMED_AWAY", 200),
        _e(BASE + 120, "a2", "contact", "ARMED_AWAY", 200),
    ]
    assert impl.solve(events) == ["DISPATCH", "DISPATCH"]


def test_boundary_verify_window_one_second_beyond():
    """Gap of verify_window_sec + 1 does NOT verify; both LOCAL_ONLY."""
    events = [
        _e(BASE, "b1", "contact", "ARMED_AWAY", 200),
        _e(BASE + 121, "b2", "contact", "ARMED_AWAY", 200),
    ]
    assert impl.solve(events) == ["LOCAL_ONLY", "LOCAL_ONLY"]


def test_armed_stay_treated_same_as_armed_away():
    """ARMED_STAY follows the same exit-delay / real-trip logic as ARMED_AWAY."""
    events = [_e(BASE, "side-door", "contact", "ARMED_STAY", 10)]
    assert impl.solve(events) == ["FALSE_ALARM"]


def test_custom_exit_delay_and_verify_window_params():
    """Custom exit_delay_sec / verify_window_sec parameters are respected."""
    events = [_e(BASE, "z1", "contact", "ARMED_AWAY", 20)]
    # with exit_delay_sec=15, 20s since armed is now a real trip (alone -> LOCAL_ONLY)
    assert impl.solve(events, exit_delay_sec=15) == ["LOCAL_ONLY"]
    # with default exit_delay_sec=45, 20s is still inside grace period
    assert impl.solve(events) == ["FALSE_ALARM"]

    events2 = [
        _e(BASE, "c1", "contact", "ARMED_AWAY", 300),
        _e(BASE + 100, "c2", "contact", "ARMED_AWAY", 300),
    ]
    # default verify_window_sec=120 -> corroborates
    assert impl.solve(events2) == ["DISPATCH", "DISPATCH"]
    # tighter verify_window_sec=50 -> does not corroborate
    assert impl.solve(events2, verify_window_sec=50) == ["LOCAL_ONLY", "LOCAL_ONLY"]


def test_panic_ignores_since_armed_and_does_not_need_corroboration():
    """A panic button trip alone in the batch, mid-exit-delay, still DISPATCHes."""
    events = [_e(BASE, "vault", "panic", "ARMED_AWAY", 2)]
    assert impl.solve(events) == ["DISPATCH"]


def test_three_zone_batch_one_isolated():
    """Two zones corroborate each other; a third isolated zone stays LOCAL_ONLY."""
    events = [
        _e(BASE, "x1", "contact", "ARMED_AWAY", 300),
        _e(BASE + 10, "x2", "glass_break", "ARMED_AWAY", 300),
        _e(BASE + 10000, "x3", "contact", "ARMED_AWAY", 300),
    ]
    assert impl.solve(events) == ["DISPATCH", "DISPATCH", "LOCAL_ONLY"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
